# Lab 10: TensorFlow with Text and Numerical Data

## Part 1: Kaggle NLP with Disaster Tweets

We will repeat the text data example, but using neural networks instead of logistic regression.

Dataset source: https://www.kaggle.com/competitions/nlp-getting-started/overview

In [1]:
import pandas as pd

import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

### Load the data

In [2]:
tweets = pd.read_csv('data/nlp-getting-started/train.csv')
tweets.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


Split the data into feature matrix and target vector:

In [3]:
X = tweets['text']
y = tweets['target']

X.shape, y.shape

((7613,), (7613,))

In [4]:
y.value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

Split the data into training and testing sets:

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=0)

### Implement Neural Network

We can create a Keras layer to implement text vectorization:

In [6]:
text_vectorizer = layers.TextVectorization(output_mode="tf-idf", ngrams=2)

In [7]:
text_vectorizer.adapt(X_train)

Now we can initialize and complile our neural network model:

In [8]:
model = tf.keras.Sequential([
  tf.keras.layers.Input(shape=(1,), dtype='string'),
  text_vectorizer,  
  tf.keras.layers.Dense(10, activation=tf.nn.relu), 
  tf.keras.layers.Dense(10, activation=tf.nn.relu),
  tf.keras.layers.Dense(2)   # Corresponds to the number of classes/labels
])

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

Fit the model to the training data:

In [9]:
%%time
num_epochs = 100   # Number of times to run through the training loop
model.fit(
    X_train,
    y_train,
    batch_size=64,
    epochs=num_epochs,
    # Suppress logging
    verbose=0,
    # Calculate validation results on 20% of the training data
    validation_split = 0.2)

CPU times: total: 2min 37s
Wall time: 3min 7s


Evaluate the model on our testing data:

In [10]:
test_loss, test_acc = model.evaluate(X_test, y_test, batch_size = 10, verbose=2)
print('\nTest accuracy:', test_acc)

153/153 - 1s - 7ms/step - accuracy: 0.7689 - loss: 1.1677

Test accuracy: 0.7688772082328796


How does this compare to the previous results?

How can we improve these results?

## Part 2: Concrete dataset

This dataset is available from the [Yellowbrick](https://www.scikit-yb.org/en/latest/api/datasets/concrete.html) library.

### Get the data
First download and import the dataset:

In [11]:
# TO DO: Import dataset
from yellowbrick.datasets import load_concrete

X, y = load_concrete()

Split the dataset into training and testing sets (use 10% for testing):

In [12]:
# TO DO: Split training and testing data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)

Now we can set up our preprocessing and neural network layers, and fit them to our data:

In [15]:
# TO DO: Set up any preprocessing layers
normalizer = tf.keras.layers.Normalization(axis=-1)

normalizer.adapt(tf.convert_to_tensor(X_train))

In [16]:
# TO DO: Initialize neural network with two layers: 10 hidden units each and sigmoid as the activation function
model = keras.Sequential([
  normalizer,
  layers.Dense(10, activation='sigmoid'),
  layers.Dense(10, activation='sigmoid'),
  layers.Dense(1)
])

# TO DO: Compile neural network (use MAE for error and Adam as the optimizer with 0.01 learning rate)
model.compile(loss='mean_absolute_error', optimizer=tf.keras.optimizers.Adam(0.01))

In [17]:
# TO DO: Fit the model to the training data (20% validation)
#%%time
model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    verbose=0, 
    epochs=100)

We can evaluate the model using the testing data:

In [18]:
# TO DO: Evaluate model using testing data
test_acc = model.evaluate(X_test, y_test, batch_size = 10, verbose=2)
print('\nTest MAE:', test_acc)

11/11 - 0s - 9ms/step - loss: 6.3382

Test MAE: 6.338212966918945


Repeat the analysis with 100 hidden units per layer:

In [19]:
# TO DO: Repeat the neural network steps with 100 hidden units per layer
model = keras.Sequential([
  normalizer,
  layers.Dense(100, activation='sigmoid'),
  layers.Dense(100, activation='sigmoid'),
  layers.Dense(1)
])

model.compile(loss='mean_absolute_error', optimizer=tf.keras.optimizers.Adam(0.01))

In [20]:
%%time
model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    verbose=0, 
    epochs=100)

CPU times: total: 10.8 s
Wall time: 22.2 s


In [21]:
test_acc = model.evaluate(X_test, y_test, batch_size = 10, verbose=2)
print('\nTest MAE:', test_acc)

11/11 - 0s - 9ms/step - loss: 4.4276

Test MAE: 4.427578449249268


Is there anything else we could try to improve the results?